# FBref Data Exploration with ScraperFC

This notebook performs an Exploratory Data Analysis (EDA) on football data using the `ScraperFC` library's FBref module. We will explore the structure of the data returned by various endpoints to understand how to best utilize it for sports analytics.

## Objectives
1. **Setup**: Initialize the FBref scraper.
2. **Season & Match Discovery**: Explore available seasons and match links.
3. **League Tables**: Analyze standings and points distribution.
4. **Stats Analysis**: Understand the structure of squad and player statistics.
5. **Match Details**: Inspect the `FBrefMatch` object and its components.

## Documentation Reference
- [ScraperFC FBref Docs](https://scraperfc.readthedocs.io/en/latest/fbref.html)

In [ ]:
import ScraperFC
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Settings
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

## 1. Initialization

We start by creating an instance of the `FBref` scraper. We'll focus on the English Premier League (EPL) for the 2023-2024 season.

In [ ]:
scraper = ScraperFC.FBref()
LEAGUE = 'EPL'
YEAR = '2023-2024'

## 2. Seasons and Matches

### 2.1 Valid Seasons
Use `get_valid_seasons` to see which years are available for the selected league.

In [ ]:
try:
    seasons = scraper.get_valid_seasons(LEAGUE)
    print(f"Available seasons for {LEAGUE}: {len(seasons)}")
    print(f"Example seasons: {list(seasons.keys())[:5]}")
except Exception as e:
    print(f"Error getting seasons: {e}")

### 2.2 Match Links
Retrieve all match URLs for the season using `get_match_links`. This list is essential for iterating through matches to scrape detailed data.

In [ ]:
try:
    match_links = scraper.get_match_links(year=YEAR, league=LEAGUE)
    print(f"Found {len(match_links)} matches for {LEAGUE} {YEAR}.")
    print("Sample Link:", match_links[0] if match_links else "None")
except Exception as e:
    print(f"Error getting match links: {e}")

## 3. League Standings

We use `scrape_league_table` to get the standings. The function returns a list of tables; the first is usually the overall league table.

In [ ]:
try:
    tables = scraper.scrape_league_table(year=YEAR, league=LEAGUE)
    if tables:
        league_table = tables[0]
        print("League Table Columns:", league_table.columns.tolist())
        display(league_table.head())
    else:
        print("No tables found.")
except Exception as e:
    print(f"Error scraping league table: {e}")

### Visualization: Points by Squad
A simple bar chart to visualize the team performance.

In [ ]:
def plot_standings(df):
    if 'Squad' not in df.columns or 'Pts' not in df.columns:
        print("Required columns 'Squad' or 'Pts' not found.")
        return
        
    plt.figure(figsize=(12, 6))
    # Sort by Points descending
    df_sorted = df.sort_values('Pts', ascending=False)
    
    sns.barplot(data=df_sorted, x='Squad', y='Pts', palette='viridis')
    plt.title(f"{LEAGUE} {YEAR} Standings", fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Points")
    plt.tight_layout()
    plt.show()

if 'league_table' in locals():
    plot_standings(league_table)

## 4. Statistics (Squad & Player)

### 4.1 Scrape Specific Category
The `scrape_stats` method returns a tuple: `(squad_stats, opponent_stats, player_stats)`. We'll inspect the 'standard' category.

In [ ]:
stat_category = 'standard'
try:
    squad_stats, opp_stats, player_stats = scraper.scrape_stats(year=YEAR, league=LEAGUE, stat_category=stat_category)
    
    print(f"--- {stat_category.upper()} STATS ---")
    print(f"Squad Stats Shape: {squad_stats.shape if squad_stats is not None else 'None'}")
    print(f"Opponent Stats Shape: {opp_stats.shape if opp_stats is not None else 'None'}")
    print(f"Player Stats Shape: {player_stats.shape if player_stats is not None else 'None'}")
    
    if player_stats is not None:
        print("\nPlayer Stats Preview:")
        display(player_stats.head())
except Exception as e:
    print(f"Error scraping stats: {e}")

### 4.2 Handling MultiIndex Columns
FBref data often comes with MultiIndex columns (e.g., ('Expected', 'xG')). We flatten them for easier analysis.

In [ ]:
def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        return ['_'.join(col).strip() if col[0] else col[1] for col in df.columns.values]
    return df.columns

if 'player_stats' in locals() and player_stats is not None:
    # Create a copy for analysis
    df_players = player_stats.copy()
    df_players.columns = flatten_columns(df_players)
    
    # Identify Goal column (usually 'Performance_Gls' or just 'Gls' depending on scrape)
    # Let's search for it
    gls_col = [c for c in df_players.columns if 'Gls' in c and 'Pen' not in c]
    if gls_col:
        col_target = gls_col[0] 
        top_scorers = df_players.nlargest(10, col_target)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=top_scorers, x=col_target, y='Player', palette='magma')
        plt.title(f"Top 10 Scorers ({LEAGUE} {YEAR})", fontsize=15)
        plt.xlabel("Goals")
        plt.show()
    else:
        print("Could not identify Goals column automatically.")
        print("Columns:", df_players.columns.tolist())

## 5. Match Data Inspection

The `scrape_match` function returns an `FBrefMatch` object. We need to inspect this object to see what attributes are available (e.g., lineups, shots, player stats).

In [ ]:
if 'match_links' in locals() and match_links:
    # Scrape the first match
    sample_link = match_links[0]
    print(f"Scraping match data from: {sample_link}")
    
    try:
        match_obj = scraper.scrape_match(sample_link)
        
        # Inspect the object using dir() to find relevant attributes
        # We filter out private attributes starting with '_'
        attributes = [attr for attr in dir(match_obj) if not attr.startswith('_')]
        print("\nAvailable attributes in FBrefMatch object:")
        print(attributes)
        
        # Let's assume some common attributes might exist or print them if found
        # The docs don't list all attributes, so this step is crucial for EDA
        
        # Check for lineups or stats
        if hasattr(match_obj, 'home_team'):
            print(f"\nHome Team: {match_obj.home_team}")
        if hasattr(match_obj, 'away_team'):
            print(f"Away Team: {match_obj.away_team}")
            
        # Try to display a dataframe if one of the attributes looks like one
        # Common attributes might include 'home_player_stats', 'shots', etc.
        for attr in ['home_player_stats', 'away_player_stats', 'summary']:
            if hasattr(match_obj, attr):
                data = getattr(match_obj, attr)
                if isinstance(data, pd.DataFrame):
                    print(f"\nPreview of '{attr}':")
                    display(data.head())
                    
    except Exception as e:
        print(f"Error scraping match: {e}")

## Summary

We have explored the primary endpoints of the ScraperFC FBref module:
1. `get_valid_seasons` and `get_match_links` for navigation.
2. `scrape_league_table` for high-level competition data.
3. `scrape_stats` for aggregated team and player metrics.
4. `scrape_match` for granular match-level data.

This foundation allows us to build more complex analysis pipelines, such as predicting match outcomes or evaluating player performance trends.